In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import tensorflow as tf
import matplotlib.pyplot as plt
import os
import yfinance

2026-03-04 17:37:31.790897: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772645852.066749      17 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772645852.139285      17 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772645852.730971      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772645852.731018      17 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772645852.731021      17 computation_placer.cc:177] computation placer alr

## Important Parameters and Tuning Setting

In [2]:
# Parameters
window_size = 120
batch_size = 30
shuffle_buffer_size = 2800
# Turn on if you want to turn learning rate
tune_learning_rate = False

## Checking and Setting Device

In [3]:
# Selecting Cuda GPU
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"
    # tf.debugging.set_log_device_placement(False) # TURN ON IF YOU WANT TO VERIFY THE GPU INFO
    print(f"Using GPU: {gpus[0].name}")
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # Disables GPU, forces CPU
    print("No CUDA GPU found — using CPU:", tf.config.list_physical_devices('CPU'))

# Confirm active device
print("Available devices:", tf.config.list_physical_devices())

No CUDA GPU found — using CPU: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]
Available devices: [PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')]


2026-03-04 17:38:06.969968: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## User defined functions that will be used throughout the program

In [4]:
def plot_data(x, y, start=0, end=None, title=None, xlabel=None, 
              ylabel=None, legend=None, ax=None):
    
    # Logic: If no ax is provided, create a new one. 
    # If one is provided, use it (this is how subplots work).
    if ax is None:
        fig, ax = plt.subplots(figsize=(10, 6))

    if isinstance(y, tuple):    
        for y_curr in y:
            ax.plot(x[start:end], y_curr[start:end], "-")
    else:
        ax.plot(x[start:end], y[start:end], "-")

    # Note: axes objects use .set_title() and .set_xlabel() 
    # instead of plt.title() and plt.xlabel()
    if title: ax.set_title(title)
    if xlabel: ax.set_xlabel(xlabel)
    if ylabel: ax.set_ylabel(ylabel)
    if legend: ax.legend(legend)
    
    ax.grid(True)


def forecast_data(model, series, window_size, batch_size):
    # have to add another axis for the feature dimension of RNN
    series = tf.expand_dims(series, axis = -1)

    # Generate a TF Dataset from the series values
    dataset = tf.data.Dataset.from_tensor_slices(series)

    # Window the data but only take those with the specified size
    dataset = dataset.window(window_size, shift=1, drop_remainder=True)

    # Flatten the windows by putting its elements in a single batch
    dataset = dataset.flat_map(lambda w: w.batch(window_size))
    
    # Create batches of windows
    dataset = dataset.batch(batch_size).prefetch(1)
    
    # Get predictions on the entire dataset
    forecast = model.predict(dataset, verbose=0)
    
    return forecast

## Getting Data From Input and Split Data

In [5]:
# Read Data from input
df = pd.read_csv('/kaggle/input/datasets/robervalt/sunspots/Sunspots.csv')
sunspot_data = df['Monthly Mean Total Sunspot Number']
time_column = list(range(len(sunspot_data)))
print(len(sunspot_data))

sunspot_data = np.array(sunspot_data)
time_column = np.array(time_column)

# Define the split time
split_time_train = 2300
split_time_valid = 2800

# Get the train set 
time_train = time_column[:split_time_train]
x_train = sunspot_data[:split_time_train]

# Get the validation set
time_valid = time_column[split_time_train:split_time_valid]
x_valid = sunspot_data[split_time_train:split_time_valid]

# Get testing set
time_test = time_column[split_time_valid:]
x_test = sunspot_data[split_time_valid:]

fig, set_axes = plt.subplots(3, 1, figsize=(10, 15))

plot_data(time_train, x_train, title="Training Set", ax=set_axes[0])
plot_data(time_valid, x_valid, title="Validation Set", ax=set_axes[1])
plot_data(time_test, x_test, title="Testing Set", ax=set_axes[2])

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/robervalt/sunspots/Sunspots.csv'

## Getting Labels from Data

In [ ]:

def windowed_dataset(series, window_size, batch_size, shuffle_buffer):
    # Add an axis for the feature dimension of RNN layers
    series = tf.expand_dims(series, axis=-1)

    # Generate a TF Dataset from the series values
    dataset = tf.data.Dataset.from_tensor_slices(series)
    
    # Window the data but only take those with the specified size
    dataset = dataset.window(window_size + 1, shift=1, drop_remainder=True)
    
    # Flatten the windows by putting its elements in a single batch
    dataset = dataset.flat_map(lambda window: window.batch(window_size + 1))

    # Create tuples with features and labels 
    dataset = dataset.map(lambda window: (window[:-1], window[-1]))
    if shuffle_buffer:
        # Shuffle the windows
        dataset = dataset.shuffle(shuffle_buffer)
    
    # Create batches of windows
    dataset = dataset.batch(batch_size)
    
    # Optimize the dataset for training
    dataset = dataset.cache().prefetch(1)
    
    return dataset

# Create Windowed Dataset
train_dataset = windowed_dataset(x_train, window_size, batch_size, shuffle_buffer_size)
valid_dataset = windowed_dataset(x_valid, window_size, batch_size, 0)

## Naive Forecast for Baseline

In [ ]:
naive_forecast = sunspot_data[split_time_valid - 1:-1]

time_step = 100

## Moving Average with Differencing for Baseline

In [ ]:
def moving_average_forecast(series, window_size):
    # Use a simple weights array for the average
    weights = np.ones(window_size) / window_size
    # 'valid' mode automatically handles the 'len(series) - window_size' logic
    forecast = np.convolve(series, weights, mode='valid')
    return forecast

#  Calculate the average on the FULL sunspot data first
full_moving_avg = moving_average_forecast(sunspot_data, 13)

moving_avg_test = full_moving_avg[split_time_valid - 12:] 

moving_avg_test = moving_avg_test[:len(x_test)]

# plot_data(time_test, (x_test, moving_avg_test), title="Moving Average Baseline")


## Model Architecture

In [ ]:
# Create the layer instance
norm_layer = tf.keras.layers.Normalization(axis=-1)
denorm_layer = tf.keras.layers.Normalization(axis=-1, invert = True)

# Calibrate it (this "bakes" the mean and std into the layer)
norm_layer.adapt(x_train.reshape(-1,1))
denorm_layer.adapt(x_train.reshape(-1,1))

print("Normalization Mean:", norm_layer.mean.numpy())
print("Normalization Variance:", norm_layer.variance.numpy())

model = tf.keras.models.Sequential([
    tf.keras.Input(shape=(window_size, 1)),
    norm_layer,
    tf.keras.layers.Conv1D(filters=60, kernel_size=3,
                      strides=1, padding="causal",
                      activation="relu"),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(60, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(60, return_sequences=True)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(60)),
    tf.keras.layers.Dense(1),
    denorm_layer
    #tf.keras.layers.Lambda(lambda x: x * 400.0) # Amplifies output back to ~300
])
model.summary()

## Tuning The Model

In [ ]:
if tune_learning_rate is True:
    # Get initial weights
    init_weights = model.get_weights()

    # Set the learning rate scheduler
    lr_schedule = tf.keras.callbacks.LearningRateScheduler(
        lambda epoch: 1e-8 * 10**(epoch / 20))

    # Initialize the optimizer
    optimizer = tf.keras.optimizers.SGD(momentum=0.9)

    # Set the training parameters
    model.compile(loss=tf.keras.losses.Huber(), optimizer=optimizer)

    # Train the model
    history = model.fit(train_dataset, epochs=100, callbacks=[lr_schedule])

    # Define the learning rate array
    lrs = 1e-8 * (10 ** (np.arange(100) / 20))

    # Set the figure size
    plt.figure(figsize=(10, 6))

    # Set the grid
    plt.grid(True)

    # Plot the loss in log scale
    plt.semilogx(lrs, history.history["loss"])

    # Increase the tickmarks size
    plt.tick_params('both', length=10, width=1, which='both')

    # Set the plot boundaries
    plt.axis([1e-8, 1e-3, 0, 50])

## Train Model

In [ ]:
# Reset states generated by Keras
tf.keras.backend.clear_session()

# Get initial weights
init_weights = model.get_weights()

# Reset the weights
model.set_weights(init_weights)

# Set the learning rate
learning_rate = 2e-4

# Set the optimizer
optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate, momentum=0.9)

# Set the training parameters
model.compile(loss=tf.keras.losses.Huber(),
              optimizer=optimizer,
              metrics=["mae"])
              
# Train the model
history = model.fit(train_dataset,
                    epochs=100,
                   validation_data=valid_dataset)

## Plot MAE and Loss

In [ ]:
# Get mae and loss from history log
mae=history.history['mae']
loss=history.history['loss']

# Get number of epochs
epochs=range(len(loss)) 

# Plot mae and loss
plot_data(
    x=epochs, 
    y=(mae, loss), 
    title='MAE and Loss', 
    xlabel='Epochs',
    legend=['MAE', 'Loss']
    )

# Only plot the last 80% of the epochs
zoom_split = int(epochs[-1] * 0.2)
epochs_zoom = epochs[zoom_split:]
mae_zoom = mae[zoom_split:]
loss_zoom = loss[zoom_split:]

# Plot zoomed mae and loss
plot_data(
    x=epochs_zoom, 
    y=(mae_zoom, loss_zoom), 
    title='MAE and Loss', 
    xlabel='Epochs',
    legend=['MAE', 'Loss']
    )

## Plot Forecast Prediction

In [ ]:
# Reduce the original series
forecast_series = sunspot_data[split_time_valid-window_size : -1] # some sizing error here 

# Use helper function to generate predictions
forecast = forecast_data(model, forecast_series, window_size, batch_size)

# Drop single dimensional axes
results = forecast.squeeze()

# Plot the results
plot_data(time_test, (x_test, results))

## Metrics - MAE and MSE

In [ ]:
## Compute the MAE and MSE for data
print(f"The MSE for Results are {tf.keras.metrics.mse(x_test, results).numpy():.3f}")
print(f"The MAE for Results are {tf.keras.metrics.mae(x_test, results).numpy():.3f}")
print()
print(f"The MSE for the Naive Forecast is {tf.keras.metrics.mse(x_test, naive_forecast).numpy():.3f}")
print(f"The MAE for the Naive Forecast is {tf.keras.metrics.mae(x_test, naive_forecast).numpy():.3f}")
print()
print(f"The MSE for the Moving Average is {tf.keras.metrics.mse(x_test, moving_avg_test).numpy():.3f}")
print(f"The MAE for the Moving Average is {tf.keras.metrics.mae(x_test, moving_avg_test).numpy():.3f}")